In [179]:
# get the data maniplulation library 
import pandas as pd
import numpy as np
import re
import math
import plotly.graph_objects as go
import plotly.express as px

from pathlib import Path

 it’s worthwhile to go back and writing up in bullet points all the sample- or language-specific considerations that were made. For instance:
~What were the overall considerations for selecting a particular language in a particular country?
~What were the more challenging exceptions we had to consider on a case by case basis?
~Why certain countries were omitted from the analyses (e.g., they didn’t have a language in the BILA dataset)
~etc.
I guess this can be organized with a table, and a different row for each of the 51 countries.
Moving forward, could you create a table with the following columns (broken down into the 36 samples):
The sample
The language
The correlation between the log count and the means
The correlation between the log count and the SDs
The correlation between the log count and the absolute distance of the means from the scale midpoint


In [180]:
Covid51countries = pd.read_csv(Path("~/Projects/hypocognition/data/raw/Covid51countries.csv").expanduser())
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,gender,ladder,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,1.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,3,0
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,2.0,7.0,3,NaN,NaN,ZH-S,snowball,AUS,3,0
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1.0,4.0,1,NaN,0.0,ZH-S,snowball,AUS,5,0
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,2.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,4,0
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1.0,5.0,1,NaN,0.0,ZH-S,snowball,AUS,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,2.0,4.0,4,NaN,0.0,EN,snowball,KEN,4,0
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,1.0,6.0,5,0.0,0.0,EN,snowball,KEN,2,0
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0


In [181]:
# We want to use on only one language of responses for each country
# Further, that language cannot be English
# Lets see what we have
country_response_count = pd.DataFrame(Covid51countries[['countryname', 'ISO3']].value_counts())
languages_per_country = Covid51countries.groupby('countryname')['language'].value_counts().reset_index(name="count")
languages_per_country = languages_per_country.merge(
    country_response_count,
    on="countryname",
    how="left"
)
languages_per_country = languages_per_country.rename(columns={"count_x": "lang_responses", "count_y": "total_responses"})
languages_per_country["percent"] = languages_per_country["lang_responses"] / languages_per_country["total_responses"] * 100

languages_per_country

,countryname,language,lang_responses,total_responses,percent
0,Australia,ZH-S,181,378,47.883598
1,Australia,EN,156,378,41.269841
2,Australia,ID,12,378,3.174603
3,Australia,ZH-T,10,378,2.645503
4,Australia,VI,5,378,1.322751
...,...,...,...,...,...
496,Vietnam,ZH-S,2,338,0.591716
497,Vietnam,FR,1,338,0.295858
498,Vietnam,JA,1,338,0.295858
499,Vietnam,PL,1,338,0.295858


In [182]:
languages_per_country['language'].unique()

array(['ZH-S', 'EN', 'ID', 'ZH-T', 'VI', 'DA', 'AR', 'ES', 'AFRI',
       'ES-ES', 'ET', 'FR', 'JA', 'SR', 'ZH-TW', 'PT-BR', 'PT', 'IT',
       'BG', 'RU', 'FA', 'UK', 'NL', 'DE', 'MS', 'TR', 'HR', 'ISL', 'NO',
       'RO', 'FI', 'KAT', 'SV', 'EL', 'CA', 'HU', 'SL', 'CS', 'PL', 'SK',
       'MAR', 'HI', 'HE', 'KO', 'KAZ', 'ML', 'MALTI', 'FILIP', 'DARI'],
      dtype=object)

In [183]:
# get a table which translates some of the language codes into names
lang_names = pd.read_csv(Path("~/Projects/hypocognition/data/external/lang_names.csv"))
languages_per_country = languages_per_country.merge(lang_names, left_on="language", right_on="Code", how="left")
languages_per_country = languages_per_country.drop(columns=["Code"])
languages_per_country

,countryname,language,lang_responses,total_responses,percent,Language_Name
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese
1,Australia,EN,156,378,41.269841,English
2,Australia,ID,12,378,3.174603,Indonesian
3,Australia,ZH-T,10,378,2.645503,Traditional Chinese
4,Australia,VI,5,378,1.322751,Vietnamese
...,...,...,...,...,...,...
496,Vietnam,ZH-S,2,338,0.591716,Simplified Chinese
497,Vietnam,FR,1,338,0.295858,French
498,Vietnam,JA,1,338,0.295858,Japanese
499,Vietnam,PL,1,338,0.295858,Polish


In [184]:
# lets just go one by one and isolate 1 language for each country
pd.set_option('display.max_rows', 100)
lpc = languages_per_country.copy()
c = lpc['countryname'].unique().tolist()
c

['Australia',
 'Brazil',
 'Bulgaria',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Croatia',
 'Curacao',
 'Denmark',
 'Egypt',
 'Finland',
 'France',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Hong Kong (S.A.R)',
 'Hungary',
 'Iceland',
 'India',
 'Indonesia',
 'Iran',
 'Ireland',
 'Israel',
 'Italy',
 'Japan',
 'Jordan',
 'Kazakhstan',
 'Kenya',
 'Malaysia',
 'Malta',
 'Mongolia',
 'Netherlands',
 'New Zealand',
 'Pakistan',
 'Peru',
 'Russia',
 'Serbia',
 'Singapore',
 'South Africa',
 'Spain',
 'Sweden',
 'Syria',
 'Taiwan',
 'Trinidad and Tobago',
 'Turkey',
 'Ukraine',
 'United Kingdom (UK)',
 'United States of America (USA)',
 'Vietnam']

In [185]:
selection = lpc.copy()
selection['Selected'] = 0
selection['reason'] = np.nan
selection

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,0,NaN
1,Australia,EN,156,378,41.269841,English,0,NaN
2,Australia,ID,12,378,3.174603,Indonesian,0,NaN
3,Australia,ZH-T,10,378,2.645503,Traditional Chinese,0,NaN
4,Australia,VI,5,378,1.322751,Vietnamese,0,NaN
...,...,...,...,...,...,...,...,...
496,Vietnam,ZH-S,2,338,0.591716,Simplified Chinese,0,NaN
497,Vietnam,FR,1,338,0.295858,French,0,NaN
498,Vietnam,JA,1,338,0.295858,Japanese,0,NaN
499,Vietnam,PL,1,338,0.295858,Polish,0,NaN


In [186]:
i = 0
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,0,NaN
1,Australia,EN,156,378,41.269841,English,0,NaN
2,Australia,ID,12,378,3.174603,Indonesian,0,NaN
3,Australia,ZH-T,10,378,2.645503,Traditional Chinese,0,NaN
4,Australia,VI,5,378,1.322751,Vietnamese,0,NaN
5,Australia,DA,3,378,0.793651,Danish,0,NaN
6,Australia,AR,2,378,0.529101,Arabic,0,NaN
7,Australia,ES,2,378,0.529101,Spanish,0,NaN
8,Australia,AFRI,1,378,0.264550,Afrikaans,0,NaN
9,Australia,ES-ES,1,378,0.264550,Spanish (Spain),0,NaN


In [187]:
# Let's use chinese for Australia, espieclly since we cant use English
# selection = selection.drop(selection.index[1:15]).reset_index(drop=True)
# selection[selection['countryname']==c[0]]
selection = selection.drop(selection.index[i+1:15])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Chinese for Australia, especially since we can't use English"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]


/tmp/ipykernel_2586964/4203792866.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' Let's use Chinese for Australia, especially since we can't use English' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  selection.loc[i, 'reason'] = " Let's use Chinese for Australia, especially since we can't use English"


,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,1,"Let's use Chinese for Australia, especially s..."


In [188]:
i = 1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
1,Brazil,PT-BR,306,325,94.153846,Brazilian Portuguese,0,NaN
2,Brazil,ES,7,325,2.153846,Spanish,0,NaN
3,Brazil,EN,3,325,0.923077,English,0,NaN
4,Brazil,DA,2,325,0.615385,Danish,0,NaN
5,Brazil,ES-ES,2,325,0.615385,Spanish (Spain),0,NaN
6,Brazil,PT,2,325,0.615385,Portuguese,0,NaN
7,Brazil,FR,1,325,0.307692,French,0,NaN
8,Brazil,IT,1,325,0.307692,Italian,0,NaN
9,Brazil,JA,1,325,0.307692,Japanese,0,NaN


In [189]:
# Let's use Brazilian Portuguese for Brazil
selection = selection.drop(selection.index[i+1:10])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "Let's use Brazilian Portuguese for Brazil"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
1,Brazil,PT-BR,306,325,94.153846,Brazilian Portuguese,1,Let's use Brazilian Portuguese for Brazil


In [190]:
i = 2
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
2,Bulgaria,BG,269,276,97.463768,Bulgarian,0,NaN
3,Bulgaria,EN,5,276,1.811594,English,0,NaN
4,Bulgaria,AR,1,276,0.362319,Arabic,0,NaN
5,Bulgaria,RU,1,276,0.362319,Russian,0,NaN


In [191]:
# 
selection = selection.drop(selection.index[i+1:6])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Bulgarian for Bulgaria"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
2,Bulgaria,BG,269,276,97.463768,Bulgarian,1,Let's use Bulgarian for Bulgaria


In [192]:
i = 3
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
3,Canada,EN,195,286,68.181818,English,0,NaN
4,Canada,ZH-S,26,286,9.090909,Simplified Chinese,0,NaN
5,Canada,FR,14,286,4.895105,French,0,NaN
6,Canada,AR,13,286,4.545455,Arabic,0,NaN
7,Canada,FA,7,286,2.447552,Persian,0,NaN
8,Canada,RU,6,286,2.097902,Russian,0,NaN
9,Canada,ES,5,286,1.748252,Spanish,0,NaN
10,Canada,UK,3,286,1.048951,Ukrainian,0,NaN
11,Canada,ZH-TW,3,286,1.048951,Traditional Chinese (Taiwan),0,NaN
12,Canada,DA,2,286,0.699301,Danish,0,NaN


In [193]:
# 
selection = selection.drop(selection.index[i+1:22])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = "Doesn't look good for Canada, we will drop them since they are primarily English "
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
3,Canada,EN,195,286,68.181818,English,0,"Doesn't look good for Canada, we will drop the..."


In [194]:
i = 4
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
4,Chile,ES,469,493,95.131846,Spanish,0,NaN
5,Chile,ES-ES,20,493,4.056795,Spanish (Spain),0,NaN
6,Chile,EN,3,493,0.608519,English,0,NaN
7,Chile,PT-BR,1,493,0.202840,Brazilian Portuguese,0,NaN


In [195]:
#  
selection = selection.drop(selection.index[i+1:8])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Spanish (General) for chile"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
4,Chile,ES,469,493,95.131846,Spanish,1,Let's use Spanish (General) for chile


In [196]:
i = 5
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
5,China,ZH-S,1988,2009,98.954704,Simplified Chinese,0,NaN
6,China,EN,10,2009,0.497760,English,0,NaN
7,China,ZH-T,7,2009,0.348432,Traditional Chinese,0,NaN
8,China,ZH-TW,2,2009,0.099552,Traditional Chinese (Taiwan),0,NaN
9,China,TR,1,2009,0.049776,Turkish,0,NaN
10,China,VI,1,2009,0.049776,Vietnamese,0,NaN


In [197]:
# 
selection = selection.drop(selection.index[i+1:11])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Simplified Chinese for china"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
5,China,ZH-S,1988,2009,98.954704,Simplified Chinese,1,Let's use Simplified Chinese for china


In [198]:
i = 6
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
6,Colombia,ES,122,237,51.476793,Spanish,0,NaN
7,Colombia,ES-ES,113,237,47.679325,Spanish (Spain),0,NaN
8,Colombia,EN,2,237,0.843882,English,0,NaN


In [199]:
# 
selection = selection.drop(selection.index[i+1: 9 ])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "Let's use Spanish (General) for Colombia  "
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
6,Colombia,ES,122,237,51.476793,Spanish,1,Let's use Spanish (General) for Colombia


In [200]:
i = 7
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
7,Croatia,HR,524,528,99.242424,Croatian,0,NaN
8,Croatia,SR,2,528,0.378788,Serbian,0,NaN
9,Croatia,DE,1,528,0.189394,German,0,NaN
10,Croatia,EN,1,528,0.189394,English,0,NaN


In [201]:
# 
selection = selection.drop(selection.index[i+1: 11])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use croatian for croatia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
7,Croatia,HR,524,528,99.242424,Croatian,1,Let's use croatian for croatia


In [202]:
i = 8
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
8,Curacao,NL,180,235,76.595745,Dutch,0,NaN
9,Curacao,EN,54,235,22.978723,English,0,NaN
10,Curacao,DE,1,235,0.425532,German,0,NaN


In [203]:
# 
selection = selection.drop(selection.index[i+1: 11])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Dutch for Curacao"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
8,Curacao,NL,180,235,76.595745,Dutch,1,Let's use Dutch for Curacao


In [204]:
i = 9
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
9,Denmark,DA,148,233,63.519313,Danish,0,NaN
10,Denmark,EN,34,233,14.592275,English,0,NaN
11,Denmark,ISL,34,233,14.592275,Icelandic,0,NaN
12,Denmark,AR,2,233,0.858369,Arabic,0,NaN
13,Denmark,FR,2,233,0.858369,French,0,NaN
14,Denmark,HR,2,233,0.858369,Croatian,0,NaN
15,Denmark,NO,2,233,0.858369,Norwegian,0,NaN
16,Denmark,RO,2,233,0.858369,Romanian,0,NaN
17,Denmark,RU,2,233,0.858369,Russian,0,NaN
18,Denmark,FI,1,233,0.429185,Finnish,0,NaN


In [205]:
# 
selection = selection.drop(selection.index[i+1: 23])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Danish for Denmark"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
9,Denmark,DA,148,233,63.519313,Danish,1,Let's use Danish for Denmark


In [206]:
i = 10
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
10,Egypt,AR,817,824,99.150485,Arabic,0,NaN
11,Egypt,EN,7,824,0.849515,English,0,NaN


In [207]:
#
selection = selection.drop(selection.index[i+1: 12])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "  Let's use Arabic for Eygpt"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
10,Egypt,AR,817,824,99.150485,Arabic,1,Let's use Arabic for Eygpt


In [208]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
11,Finland,FI,260,289,89.965398,Finnish,0,NaN
12,Finland,EN,14,289,4.844291,English,0,NaN
13,Finland,SV,7,289,2.422145,Swedish,0,NaN
14,Finland,UK,2,289,0.692042,Ukrainian,0,NaN
15,Finland,AR,1,289,0.346021,Arabic,0,NaN
16,Finland,DA,1,289,0.346021,Danish,0,NaN
17,Finland,ET,1,289,0.346021,Estonian,0,NaN
18,Finland,ID,1,289,0.346021,Indonesian,0,NaN
19,Finland,VI,1,289,0.346021,Vietnamese,0,NaN
20,Finland,ZH-TW,1,289,0.346021,Traditional Chinese (Taiwan),0,NaN


In [209]:
# 
selection = selection.drop(selection.index[i+1: 21])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Finnish for Finland"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
11,Finland,FI,260,289,89.965398,Finnish,1,Let's use Finnish for Finland


In [210]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
12,France,FR,257,363,70.798898,French,0,NaN
13,France,AR,24,363,6.611570,Arabic,0,NaN
14,France,EN,23,363,6.336088,English,0,NaN
15,France,DA,8,363,2.203857,Danish,0,NaN
16,France,ZH-S,5,363,1.377410,Simplified Chinese,0,NaN
17,France,DE,4,363,1.101928,German,0,NaN
18,France,RU,4,363,1.101928,Russian,0,NaN
19,France,BG,3,363,0.826446,Bulgarian,0,NaN
20,France,EL,3,363,0.826446,Greek,0,NaN
21,France,NL,3,363,0.826446,Dutch,0,NaN


In [211]:
# 
selection = selection.drop(selection.index[i+1: 38])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use french for france"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
12,France,FR,257,363,70.798898,French,1,Let's use french for france


In [212]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
13,Georgia,KAT,234,236,99.152542,Georgian,0,NaN
14,Georgia,EN,2,236,0.847458,English,0,NaN


In [213]:
#
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "  Let's use Georgian for Georgia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
13,Georgia,KAT,234,236,99.152542,Georgian,1,Let's use Georgian for Georgia


In [214]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
14,Germany,DE,291,567,51.322751,German,0,NaN
15,Germany,AR,123,567,21.693122,Arabic,0,NaN
16,Germany,EN,54,567,9.523810,English,0,NaN
17,Germany,ZH-S,12,567,2.116402,Simplified Chinese,0,NaN
18,Germany,ES,9,567,1.587302,Spanish,0,NaN
19,Germany,HR,8,567,1.410935,Croatian,0,NaN
20,Germany,DA,7,567,1.234568,Danish,0,NaN
21,Germany,FA,6,567,1.058201,Persian,0,NaN
22,Germany,RU,6,567,1.058201,Russian,0,NaN
23,Germany,SR,6,567,1.058201,Serbian,0,NaN


In [215]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use German for Germany"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
14,Germany,DE,291,567,51.322751,German,1,Let's use German for Germany


In [216]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
15,Ghana,EN,327,328,99.695122,English,0,NaN
16,Ghana,ZH-S,1,328,0.304878,Simplified Chinese,0,NaN


In [217]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " doesnt look good for Ghana, well drop it"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
15,Ghana,EN,327,328,99.695122,English,0,"doesnt look good for Ghana, well drop it"


In [218]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
16,Greece,EL,227,253,89.723320,Greek,0,NaN
17,Greece,SR,8,253,3.162055,Serbian,0,NaN
18,Greece,EN,7,253,2.766798,English,0,NaN
19,Greece,RO,2,253,0.790514,Romanian,0,NaN
20,Greece,AR,1,253,0.395257,Arabic,0,NaN
21,Greece,BG,1,253,0.395257,Bulgarian,0,NaN
22,Greece,DA,1,253,0.395257,Danish,0,NaN
23,Greece,FA,1,253,0.395257,Persian,0,NaN
24,Greece,NO,1,253,0.395257,Norwegian,0,NaN
25,Greece,PT-BR,1,253,0.395257,Brazilian Portuguese,0,NaN


In [219]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Greek for Greece"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
16,Greece,EL,227,253,89.72332,Greek,1,Let's use Greek for Greece


In [220]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
17,Hong Kong (S.A.R),ZH-T,139,228,60.964912,Traditional Chinese,0,NaN
18,Hong Kong (S.A.R),ZH-S,55,228,24.122807,Simplified Chinese,0,NaN
19,Hong Kong (S.A.R),EN,20,228,8.771930,English,0,NaN
20,Hong Kong (S.A.R),ZH-TW,12,228,5.263158,Traditional Chinese (Taiwan),0,NaN
21,Hong Kong (S.A.R),DA,1,228,0.438596,Danish,0,NaN
22,Hong Kong (S.A.R),FR,1,228,0.438596,French,0,NaN


In [221]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Traditional Chinese for Hong Kong"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
17,Hong Kong (S.A.R),ZH-T,139,228,60.964912,Traditional Chinese,1,Let's use Traditional Chinese for Hong Kong


In [222]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
18,Hungary,HU,256,464,55.172414,Hungarian,0,NaN
19,Hungary,EN,155,464,33.405172,English,0,NaN
20,Hungary,AR,13,464,2.801724,Arabic,0,NaN
21,Hungary,ZH-S,9,464,1.939655,Simplified Chinese,0,NaN
22,Hungary,VI,5,464,1.077586,Vietnamese,0,NaN
23,Hungary,RU,4,464,0.862069,Russian,0,NaN
24,Hungary,ES-ES,3,464,0.646552,Spanish (Spain),0,NaN
25,Hungary,PT-BR,3,464,0.646552,Brazilian Portuguese,0,NaN
26,Hungary,FR,2,464,0.431034,French,0,NaN
27,Hungary,JA,2,464,0.431034,Japanese,0,NaN


In [223]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Hungarian for Hungary"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
18,Hungary,HU,256,464,55.172414,Hungarian,1,Let's use Hungarian for Hungary


In [224]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
19,Iceland,ISL,348,354,98.305085,Icelandic,0,NaN
20,Iceland,EN,6,354,1.694915,English,0,NaN


In [225]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Icelandic for Iceland"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
19,Iceland,ISL,348,354,98.305085,Icelandic,1,Let's use Icelandic for Iceland


In [226]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
20,India,EN,153,243,62.962963,English,0,NaN
21,India,MAR,56,243,23.045267,Marathi,0,NaN
22,India,HI,31,243,12.757202,Hindi,0,NaN
23,India,DA,2,243,0.823045,Danish,0,NaN
24,India,NL,1,243,0.411523,Dutch,0,NaN


In [227]:
# This is a tough call. I think we will use Maranthi, despite it being pretty small
selection = selection.drop(selection.index[i]).reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
20,India,MAR,56,243,23.045267,Marathi,0,NaN
21,India,HI,31,243,12.757202,Hindi,0,NaN
22,India,DA,2,243,0.823045,Danish,0,NaN
23,India,NL,1,243,0.411523,Dutch,0,NaN


In [228]:
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " This is a tough call. I think we will use Maranthi, despite it being pretty small"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
20,India,MAR,56,243,23.045267,Marathi,1,This is a tough call. I think we will use Mar...


In [229]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
21,Indonesia,ID,648,656,98.780488,Indonesian,0,NaN
22,Indonesia,EN,4,656,0.609756,English,0,NaN
23,Indonesia,NL,2,656,0.304878,Dutch,0,NaN
24,Indonesia,DA,1,656,0.152439,Danish,0,NaN
25,Indonesia,FA,1,656,0.152439,Persian,0,NaN


In [230]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Indonesian for indonesia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
21,Indonesia,ID,648,656,98.780488,Indonesian,1,Let's use Indonesian for indonesia


In [231]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
22,Iran,FA,187,195,95.897436,Persian,0,NaN
23,Iran,EN,7,195,3.589744,English,0,NaN
24,Iran,FR,1,195,0.512821,French,0,NaN


In [232]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use persian for Iran"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
22,Iran,FA,187,195,95.897436,Persian,1,Let's use persian for Iran


In [233]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
23,Ireland,EN,351,362,96.961326,English,0,NaN
24,Ireland,HR,3,362,0.828729,Croatian,0,NaN
25,Ireland,PL,2,362,0.552486,Polish,0,NaN
26,Ireland,RO,2,362,0.552486,Romanian,0,NaN
27,Ireland,RU,2,362,0.552486,Russian,0,NaN
28,Ireland,AR,1,362,0.276243,Arabic,0,NaN
29,Ireland,FI,1,362,0.276243,Finnish,0,NaN


In [234]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " Doesn't look good for Ireland, their primary language is definitely Elnglish. we will drop them"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
23,Ireland,EN,351,362,96.961326,English,0,"Doesn't look good for Ireland, their primary ..."


In [235]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
24,Israel,HE,151,211,71.563981,Hebrew,0,NaN
25,Israel,EN,46,211,21.800948,English,0,NaN
26,Israel,RU,7,211,3.317536,Russian,0,NaN
27,Israel,DA,2,211,0.947867,Danish,0,NaN
28,Israel,ES,1,211,0.473934,Spanish,0,NaN
29,Israel,ES-ES,1,211,0.473934,Spanish (Spain),0,NaN
30,Israel,PL,1,211,0.473934,Polish,0,NaN
31,Israel,SV,1,211,0.473934,Swedish,0,NaN
32,Israel,UK,1,211,0.473934,Ukrainian,0,NaN


In [236]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Lets use Hebrew for Israel"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
24,Israel,HE,151,211,71.563981,Hebrew,1,Lets use Hebrew for Israel


In [237]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
25,Italy,IT,426,466,91.416309,Italian,0,NaN
26,Italy,EN,12,466,2.575107,English,0,NaN
27,Italy,ZH-S,7,466,1.502146,Simplified Chinese,0,NaN
28,Italy,AR,2,466,0.429185,Arabic,0,NaN
29,Italy,EL,2,466,0.429185,Greek,0,NaN
30,Italy,RU,2,466,0.429185,Russian,0,NaN
31,Italy,SR,2,466,0.429185,Serbian,0,NaN
32,Italy,SV,2,466,0.429185,Swedish,0,NaN
33,Italy,TR,2,466,0.429185,Turkish,0,NaN
34,Italy,DA,1,466,0.214592,Danish,0,NaN


In [238]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " lets use Italian for Italy"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
25,Italy,IT,426,466,91.416309,Italian,1,lets use Italian for Italy


In [239]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
26,Japan,JA,1288,1352,95.266272,Japanese,0,NaN
27,Japan,ZH-S,22,1352,1.627219,Simplified Chinese,0,NaN
28,Japan,EN,20,1352,1.479290,English,0,NaN
29,Japan,ID,6,1352,0.443787,Indonesian,0,NaN
30,Japan,ZH-TW,5,1352,0.369822,Traditional Chinese (Taiwan),0,NaN
31,Japan,VI,3,1352,0.221893,Vietnamese,0,NaN
32,Japan,DA,2,1352,0.147929,Danish,0,NaN
33,Japan,KO,2,1352,0.147929,Korean,0,NaN
34,Japan,ES,1,1352,0.073964,Spanish,0,NaN
35,Japan,FR,1,1352,0.073964,French,0,NaN


In [240]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Japanese for Japan"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
26,Japan,JA,1288,1352,95.266272,Japanese,1,Let's use Japanese for Japan


In [241]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
27,Jordan,AR,233,236,98.728814,Arabic,0,NaN
28,Jordan,EN,2,236,0.847458,English,0,NaN
29,Jordan,DA,1,236,0.423729,Danish,0,NaN


In [242]:
#
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "  Let's use Arabic for Jordan"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
27,Jordan,AR,233,236,98.728814,Arabic,1,Let's use Arabic for Jordan


In [243]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
28,Kazakhstan,RU,279,291,95.876289,Russian,0,NaN
29,Kazakhstan,EN,8,291,2.749141,English,0,NaN
30,Kazakhstan,KAZ,4,291,1.374570,Kazakh,0,NaN


In [244]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Russian for Kazakhstan"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
28,Kazakhstan,RU,279,291,95.876289,Russian,1,Let's use Russian for Kazakhstan


In [245]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
29,Kenya,EN,207,207,100.0,English,0,NaN


In [246]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = "Nope, sorry Kenya, to much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
29,Kenya,EN,207,207,100.0,English,0,"Nope, sorry Kenya, to much english"


In [247]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
30,Malaysia,MS,155,214,72.429907,Malay,0,NaN
31,Malaysia,EN,51,214,23.831776,English,0,NaN
32,Malaysia,ID,3,214,1.401869,Indonesian,0,NaN
33,Malaysia,AR,1,214,0.467290,Arabic,0,NaN
34,Malaysia,ES-ES,1,214,0.467290,Spanish (Spain),0,NaN
35,Malaysia,ML,1,214,0.467290,Malayalam,0,NaN
36,Malaysia,VI,1,214,0.467290,Vietnamese,0,NaN
37,Malaysia,ZH-S,1,214,0.467290,Simplified Chinese,0,NaN


In [248]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use Malay for Malaysia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
30,Malaysia,MS,155,214,72.429907,Malay,1,Let's use Malay for Malaysia


In [249]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
31,Malta,EN,980,1290,75.968992,English,0,NaN
32,Malta,MALTI,284,1290,22.015504,Maltese,0,NaN
33,Malta,IT,4,1290,0.310078,Italian,0,NaN
34,Malta,EL,3,1290,0.232558,Greek,0,NaN
35,Malta,HU,3,1290,0.232558,Hungarian,0,NaN
36,Malta,RU,3,1290,0.232558,Russian,0,NaN
37,Malta,DE,2,1290,0.155039,German,0,NaN
38,Malta,FR,2,1290,0.155039,French,0,NaN
39,Malta,NL,2,1290,0.155039,Dutch,0,NaN
40,Malta,AR,1,1290,0.077519,Arabic,0,NaN


In [250]:
# 
selection = selection.drop(selection.index[i]).reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
31,Malta,MALTI,284,1290,22.015504,Maltese,0,NaN
32,Malta,IT,4,1290,0.310078,Italian,0,NaN
33,Malta,EL,3,1290,0.232558,Greek,0,NaN
34,Malta,HU,3,1290,0.232558,Hungarian,0,NaN
35,Malta,RU,3,1290,0.232558,Russian,0,NaN
36,Malta,DE,2,1290,0.155039,German,0,NaN
37,Malta,FR,2,1290,0.155039,French,0,NaN
38,Malta,NL,2,1290,0.155039,Dutch,0,NaN
39,Malta,AR,1,1290,0.077519,Arabic,0,NaN
40,Malta,BG,1,1290,0.077519,Bulgarian,0,NaN


In [251]:
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Tough call, we will use Maltese for Malta instead of english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
31,Malta,MALTI,284,1290,22.015504,Maltese,1,"Tough call, we will use Maltese for Malta ins..."


In [252]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
32,Mongolia,EN,188,208,90.384615,English,0,NaN
33,Mongolia,RU,13,208,6.250000,Russian,0,NaN
34,Mongolia,KO,2,208,0.961538,Korean,0,NaN
35,Mongolia,TR,2,208,0.961538,Turkish,0,NaN
36,Mongolia,CS,1,208,0.480769,Czech,0,NaN
37,Mongolia,DE,1,208,0.480769,German,0,NaN
38,Mongolia,JA,1,208,0.480769,Japanese,0,NaN


In [253]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " Nope, sorry mongolia, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
32,Mongolia,EN,188,208,90.384615,English,0,"Nope, sorry mongolia, too much english"


In [254]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
33,Netherlands,NL,524,1348,38.872404,Dutch,0,NaN
34,Netherlands,EN,201,1348,14.910979,English,0,NaN
35,Netherlands,AR,144,1348,10.682493,Arabic,0,NaN
36,Netherlands,EL,137,1348,10.163205,Greek,0,NaN
37,Netherlands,FA,82,1348,6.083086,Persian,0,NaN
38,Netherlands,DA,45,1348,3.338279,Danish,0,NaN
39,Netherlands,ZH-TW,35,1348,2.596439,Traditional Chinese (Taiwan),0,NaN
40,Netherlands,ZH-S,30,1348,2.225519,Simplified Chinese,0,NaN
41,Netherlands,SV,24,1348,1.780415,Swedish,0,NaN
42,Netherlands,VI,17,1348,1.261128,Vietnamese,0,NaN


In [255]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Let's use dutch for the Netherlands"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
33,Netherlands,NL,524,1348,38.872404,Dutch,1,Let's use dutch for the Netherlands


In [256]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
34,New Zealand,EN,150,262,57.251908,English,0,NaN
35,New Zealand,ZH-S,102,262,38.931298,Simplified Chinese,0,NaN
36,New Zealand,DA,2,262,0.763359,Danish,0,NaN
37,New Zealand,KO,2,262,0.763359,Korean,0,NaN
38,New Zealand,VI,2,262,0.763359,Vietnamese,0,NaN
39,New Zealand,ZH-T,2,262,0.763359,Traditional Chinese,0,NaN
40,New Zealand,JA,1,262,0.381679,Japanese,0,NaN
41,New Zealand,RU,1,262,0.381679,Russian,0,NaN


In [257]:
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " We're actually not going to keep Australia and New Zealand in for now, perhaps we will use them for robustness testing since they have such high percentages of chinese response scores"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
34,New Zealand,EN,150,262,57.251908,English,0,We're actually not going to keep Australia an...


In [258]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
35,Pakistan,EN,555,557,99.640934,English,0,NaN
36,Pakistan,FR,2,557,0.359066,French,0,NaN


In [259]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " nope, sorry pakistan, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
35,Pakistan,EN,555,557,99.640934,English,0,"nope, sorry pakistan, too much english"


In [260]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
36,Peru,ES-ES,215,357,60.224090,Spanish (Spain),0,NaN
37,Peru,ES,128,357,35.854342,Spanish,0,NaN
38,Peru,EN,13,357,3.641457,English,0,NaN
39,Peru,IT,1,357,0.280112,Italian,0,NaN


In [261]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " lets use Spanish spain for PEru"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
36,Peru,ES-ES,215,357,60.22409,Spanish (Spain),1,lets use Spanish spain for PEru


In [262]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
37,Russia,RU,577,586,98.464164,Russian,0,NaN
38,Russia,EN,6,586,1.023891,English,0,NaN
39,Russia,AR,1,586,0.170648,Arabic,0,NaN
40,Russia,ES-ES,1,586,0.170648,Spanish (Spain),0,NaN
41,Russia,RO,1,586,0.170648,Romanian,0,NaN


In [263]:

selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " # lets use russian for russia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
37,Russia,RU,577,586,98.464164,Russian,1,# lets use russian for russia


In [264]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
38,Serbia,SR,351,354,99.152542,Serbian,0,NaN
39,Serbia,DA,1,354,0.282486,Danish,0,NaN
40,Serbia,EN,1,354,0.282486,English,0,NaN
41,Serbia,HR,1,354,0.282486,Croatian,0,NaN


In [265]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " lets use serbian for serbia"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
38,Serbia,SR,351,354,99.152542,Serbian,1,lets use serbian for serbia


In [266]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
39,Singapore,EN,187,208,89.903846,English,0,NaN
40,Singapore,ZH-S,8,208,3.846154,Simplified Chinese,0,NaN
41,Singapore,DA,3,208,1.442308,Danish,0,NaN
42,Singapore,ID,2,208,0.961538,Indonesian,0,NaN
43,Singapore,MS,2,208,0.961538,Malay,0,NaN
44,Singapore,ZH-TW,2,208,0.961538,Traditional Chinese (Taiwan),0,NaN
45,Singapore,AR,1,208,0.480769,Arabic,0,NaN
46,Singapore,FA,1,208,0.480769,Persian,0,NaN
47,Singapore,JA,1,208,0.480769,Japanese,0,NaN
48,Singapore,PL,1,208,0.480769,Polish,0,NaN


In [267]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " nope, sorry singapore, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
39,Singapore,EN,187,208,89.903846,English,0,"nope, sorry singapore, too much english"


In [268]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
40,South Africa,EN,302,426,70.892019,English,0,NaN
41,South Africa,AFRI,112,426,26.291080,Afrikaans,0,NaN
42,South Africa,FR,7,426,1.643192,French,0,NaN
43,South Africa,AR,2,426,0.469484,Arabic,0,NaN
44,South Africa,BG,1,426,0.234742,Bulgarian,0,NaN
45,South Africa,DE,1,426,0.234742,German,0,NaN
46,South Africa,NL,1,426,0.234742,Dutch,0,NaN


In [269]:
selection = selection.drop(selection.index[i]).reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
40,South Africa,AFRI,112,426,26.291080,Afrikaans,0,NaN
41,South Africa,FR,7,426,1.643192,French,0,NaN
42,South Africa,AR,2,426,0.469484,Arabic,0,NaN
43,South Africa,BG,1,426,0.234742,Bulgarian,0,NaN
44,South Africa,DE,1,426,0.234742,German,0,NaN
45,South Africa,NL,1,426,0.234742,Dutch,0,NaN


In [270]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " Interesting, we will use Africaans for south africa even though it only has 26%"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
40,South Africa,AFRI,112,426,26.29108,Afrikaans,1,"Interesting, we will use Africaans for south ..."


In [271]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
41,Spain,CA,187,418,44.736842,Catalan,0,NaN
42,Spain,ES-ES,105,418,25.119617,Spanish (Spain),0,NaN
43,Spain,ZH-S,61,418,14.593301,Simplified Chinese,0,NaN
44,Spain,ES,21,418,5.023923,Spanish,0,NaN
45,Spain,EN,9,418,2.153110,English,0,NaN
46,Spain,ISL,5,418,1.196172,Icelandic,0,NaN
47,Spain,DA,4,418,0.956938,Danish,0,NaN
48,Spain,AR,3,418,0.717703,Arabic,0,NaN
49,Spain,BG,3,418,0.717703,Bulgarian,0,NaN
50,Spain,NL,3,418,0.717703,Dutch,0,NaN


In [272]:
selection = selection.drop(selection.index[i]).reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
41,Spain,ES-ES,105,418,25.119617,Spanish (Spain),0,NaN
42,Spain,ZH-S,61,418,14.593301,Simplified Chinese,0,NaN
43,Spain,ES,21,418,5.023923,Spanish,0,NaN
44,Spain,EN,9,418,2.153110,English,0,NaN
45,Spain,ISL,5,418,1.196172,Icelandic,0,NaN
46,Spain,DA,4,418,0.956938,Danish,0,NaN
47,Spain,AR,3,418,0.717703,Arabic,0,NaN
48,Spain,BG,3,418,0.717703,Bulgarian,0,NaN
49,Spain,NL,3,418,0.717703,Dutch,0,NaN
50,Spain,ET,2,418,0.478469,Estonian,0,NaN


In [273]:
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " I know we don't have Catalan dictionary, so we will pre-emptively use Spanish Spain"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
41,Spain,ES-ES,105,418,25.119617,Spanish (Spain),1,"I know we don't have Catalan dictionary, so w..."


In [274]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
42,Sweden,SV,400,529,75.614367,Swedish,0,NaN
43,Sweden,EN,37,529,6.994329,English,0,NaN
44,Sweden,AR,23,529,4.347826,Arabic,0,NaN
45,Sweden,DA,21,529,3.969754,Danish,0,NaN
46,Sweden,NL,20,529,3.780718,Dutch,0,NaN
47,Sweden,ISL,6,529,1.134216,Icelandic,0,NaN
48,Sweden,CS,2,529,0.378072,Czech,0,NaN
49,Sweden,ES,2,529,0.378072,Spanish,0,NaN
50,Sweden,FI,2,529,0.378072,Finnish,0,NaN
51,Sweden,ID,2,529,0.378072,Indonesian,0,NaN


In [275]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " We will use Swedish for Sweden"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
42,Sweden,SV,400,529,75.614367,Swedish,1,We will use Swedish for Sweden


In [276]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
43,Syria,AR,247,248,99.596774,Arabic,0,NaN
44,Syria,SR,1,248,0.403226,Serbian,0,NaN


In [277]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " we will use Arabic for Syria"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
43,Syria,AR,247,248,99.596774,Arabic,1,we will use Arabic for Syria


In [278]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
44,Taiwan,ZH-TW,747,761,98.160315,Traditional Chinese (Taiwan),0,NaN
45,Taiwan,EN,5,761,0.657030,English,0,NaN
46,Taiwan,ZH-S,4,761,0.525624,Simplified Chinese,0,NaN
47,Taiwan,ZH-T,2,761,0.262812,Traditional Chinese,0,NaN
48,Taiwan,DA,1,761,0.131406,Danish,0,NaN
49,Taiwan,ID,1,761,0.131406,Indonesian,0,NaN
50,Taiwan,VI,1,761,0.131406,Vietnamese,0,NaN


In [279]:
#
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = "  we will use Traditional Chinese (Taiwan) for Taiwan"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
44,Taiwan,ZH-TW,747,761,98.160315,Traditional Chinese (Taiwan),1,we will use Traditional Chinese (Taiwan) for...


In [280]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
45,Trinidad and Tobago,EN,333,334,99.700599,English,0,NaN
46,Trinidad and Tobago,ES,1,334,0.299401,Spanish,0,NaN


In [281]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " nope, sorry T&T, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
45,Trinidad and Tobago,EN,333,334,99.700599,English,0,"nope, sorry T&T, too much english"


In [282]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
46,Turkey,TR,253,346,73.121387,Turkish,0,NaN
47,Turkey,AR,77,346,22.254335,Arabic,0,NaN
48,Turkey,EN,11,346,3.179191,English,0,NaN
49,Turkey,FA,2,346,0.578035,Persian,0,NaN
50,Turkey,DA,1,346,0.289017,Danish,0,NaN
51,Turkey,ID,1,346,0.289017,Indonesian,0,NaN
52,Turkey,RO,1,346,0.289017,Romanian,0,NaN


In [283]:

selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " # we will use Turkish for turkey"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
46,Turkey,TR,253,346,73.121387,Turkish,1,# we will use Turkish for turkey


In [284]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
47,Ukraine,UK,657,698,94.126074,Ukrainian,0,NaN
48,Ukraine,RU,34,698,4.871060,Russian,0,NaN
49,Ukraine,EN,4,698,0.573066,English,0,NaN
50,Ukraine,AR,1,698,0.143266,Arabic,0,NaN
51,Ukraine,DARI,1,698,0.143266,Dari,0,NaN
52,Ukraine,FA,1,698,0.143266,Persian,0,NaN


In [285]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " we will use ukrainian for Ukraine"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
47,Ukraine,UK,657,698,94.126074,Ukrainian,1,we will use ukrainian for Ukraine


In [286]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
48,United Kingdom (UK),EN,454,662,68.580060,English,0,NaN
49,United Kingdom (UK),ZH-S,34,662,5.135952,Simplified Chinese,0,NaN
50,United Kingdom (UK),HU,24,662,3.625378,Hungarian,0,NaN
51,United Kingdom (UK),ID,20,662,3.021148,Indonesian,0,NaN
52,United Kingdom (UK),DA,16,662,2.416918,Danish,0,NaN
53,United Kingdom (UK),EL,15,662,2.265861,Greek,0,NaN
54,United Kingdom (UK),AR,14,662,2.114804,Arabic,0,NaN
55,United Kingdom (UK),PL,11,662,1.661631,Polish,0,NaN
56,United Kingdom (UK),FR,9,662,1.359517,French,0,NaN
57,United Kingdom (UK),BG,7,662,1.057402,Bulgarian,0,NaN


In [287]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " nope, sorry UK, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
48,United Kingdom (UK),EN,454,662,68.58006,English,0,"nope, sorry UK, too much english"


In [288]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
49,United States of America (USA),EN,768,952,80.672269,English,0,NaN
50,United States of America (USA),ZH-S,45,952,4.726891,Simplified Chinese,0,NaN
51,United States of America (USA),ZH-TW,28,952,2.941176,Traditional Chinese (Taiwan),0,NaN
52,United States of America (USA),ES,15,952,1.575630,Spanish,0,NaN
53,United States of America (USA),JA,13,952,1.365546,Japanese,0,NaN
54,United States of America (USA),DA,10,952,1.050420,Danish,0,NaN
55,United States of America (USA),ES-ES,10,952,1.050420,Spanish (Spain),0,NaN
56,United States of America (USA),KO,9,952,0.945378,Korean,0,NaN
57,United States of America (USA),VI,6,952,0.630252,Vietnamese,0,NaN
58,United States of America (USA),AR,5,952,0.525210,Arabic,0,NaN


In [289]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 0
selection.loc[i, 'reason'] = " nope, sorry US, too much english"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
49,United States of America (USA),EN,768,952,80.672269,English,0,"nope, sorry US, too much english"


In [290]:
i = i+1
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
50,Vietnam,VI,292,338,86.390533,Vietnamese,0,NaN
51,Vietnam,EN,37,338,10.946746,English,0,NaN
52,Vietnam,DA,3,338,0.887574,Danish,0,NaN
53,Vietnam,ZH-S,2,338,0.591716,Simplified Chinese,0,NaN
54,Vietnam,FR,1,338,0.295858,French,0,NaN
55,Vietnam,JA,1,338,0.295858,Japanese,0,NaN
56,Vietnam,PL,1,338,0.295858,Polish,0,NaN
57,Vietnam,ZH-TW,1,338,0.295858,Traditional Chinese (Taiwan),0,NaN


In [291]:
# 
selection = selection.drop(selection.index[i+1: max(selection[selection['countryname']==c[i]].index) +1])
selection.loc[i,'Selected'] = 1
selection.loc[i, 'reason'] = " We will use vietnamese for vietnam"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[i]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
50,Vietnam,VI,292,338,86.390533,Vietnamese,1,We will use vietnamese for vietnam


In [292]:
i = i+1
selection[selection['countryname']==c[i]]

IndexError: list index out of range

In [293]:
selection

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,1,"Let's use Chinese for Australia, especially s..."
1,Brazil,PT-BR,306,325,94.153846,Brazilian Portuguese,1,Let's use Brazilian Portuguese for Brazil
2,Bulgaria,BG,269,276,97.463768,Bulgarian,1,Let's use Bulgarian for Bulgaria
3,Canada,EN,195,286,68.181818,English,0,"Doesn't look good for Canada, we will drop the..."
4,Chile,ES,469,493,95.131846,Spanish,1,Let's use Spanish (General) for chile
5,China,ZH-S,1988,2009,98.954704,Simplified Chinese,1,Let's use Simplified Chinese for china
6,Colombia,ES,122,237,51.476793,Spanish,1,Let's use Spanish (General) for Colombia
7,Croatia,HR,524,528,99.242424,Croatian,1,Let's use croatian for croatia
8,Curacao,NL,180,235,76.595745,Dutch,1,Let's use Dutch for Curacao
9,Denmark,DA,148,233,63.519313,Danish,1,Let's use Danish for Denmark


In [294]:
# 
selection.loc[0,'Selected'] = 0
selection.loc[0, 'reason'] = " We will leave out Australia for now because it has only English and chinese. Perhaps we will use the chinese respobse for robustness testing"
selection = selection.reset_index(drop=True)
selection[selection['countryname']==c[0]]

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...


In [295]:
# Lets filter the dataset using our new selection
filtered = Covid51countries.merge(
    selection,
    on=['countryname', 'language'],
    how='inner'
)
filtered

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,datasource,ISO3,longstring,na_count,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,snowball,AUS,3,0,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,snowball,AUS,3,0,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,snowball,AUS,5,0,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,snowball,AUS,4,0,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,snowball,AUS,3,0,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18913,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,snowball,KEN,4,0,207,207,100.000000,English,0,"Nope, sorry Kenya, to much english"
18914,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,snowball,KEN,2,0,207,207,100.000000,English,0,"Nope, sorry Kenya, to much english"
18915,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,snowball,KEN,3,0,207,207,100.000000,English,0,"Nope, sorry Kenya, to much english"
18916,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,snowball,KEN,3,0,207,207,100.000000,English,0,"Nope, sorry Kenya, to much english"


In [296]:
# use only selected languages from selected countries
filtered = filtered[filtered['Selected']==1].reset_index(drop=True)
filtered

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,datasource,ISO3,longstring,na_count,lang_responses,total_responses,percent,Language_Name,Selected,reason
0,109,Malaysia,21/04/2020 19:11,21/04/2020 19:34,0.0,100.0,1373.0,1.0,21/04/2020 19:34,97.0,...,snowball,MYS,9,0,155,214,72.429907,Malay,1,Let's use Malay for Malaysia
1,109,Malaysia,21/04/2020 06:27,21/04/2020 06:46,0.0,100.0,1135.0,1.0,21/04/2020 06:46,100.0,...,snowball,MYS,3,0,155,214,72.429907,Malay,1,Let's use Malay for Malaysia
2,109,Malaysia,20/04/2020 21:14,20/04/2020 21:29,0.0,100.0,951.0,1.0,20/04/2020 21:29,NaN,...,snowball,MYS,5,0,155,214,72.429907,Malay,1,Let's use Malay for Malaysia
3,109,Malaysia,21/04/2020 02:24,21/04/2020 02:37,0.0,100.0,745.0,1.0,21/04/2020 02:37,100.0,...,snowball,MYS,2,0,155,214,72.429907,Malay,1,Let's use Malay for Malaysia
4,109,Malaysia,20/04/2020 22:00,20/04/2020 22:09,0.0,100.0,538.0,1.0,20/04/2020 22:09,100.0,...,snowball,MYS,4,0,155,214,72.429907,Malay,1,Let's use Malay for Malaysia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15017,91,Kazakhstan,25/04/2020 13:36,25/04/2020 13:44,0.0,100.0,470.0,1.0,25/04/2020 13:44,100.0,...,snowball,KAZ,3,0,279,291,95.876289,Russian,1,Let's use Russian for Kazakhstan
15018,91,Kazakhstan,06/05/2020 04:03,06/05/2020 04:18,0.0,100.0,884.0,1.0,06/05/2020 04:18,100.0,...,snowball,KAZ,3,0,279,291,95.876289,Russian,1,Let's use Russian for Kazakhstan
15019,91,Kazakhstan,06/05/2020 12:32,06/05/2020 12:41,0.0,100.0,573.0,1.0,06/05/2020 12:41,100.0,...,snowball,KAZ,4,0,279,291,95.876289,Russian,1,Let's use Russian for Kazakhstan
15020,91,Kazakhstan,06/05/2020 10:37,06/05/2020 10:49,0.0,100.0,701.0,1.0,06/05/2020 10:49,100.0,...,snowball,KAZ,5,0,279,291,95.876289,Russian,1,Let's use Russian for Kazakhstan


In [297]:
# Check to make sure it worked
filtered.groupby('countryname')['Language_Name'].unique()

countryname
Brazil                       [Brazilian Portuguese]
Bulgaria                                [Bulgarian]
Chile                                     [Spanish]
China                          [Simplified Chinese]
Colombia                                  [Spanish]
Croatia                                  [Croatian]
Curacao                                     [Dutch]
Denmark                                    [Danish]
Egypt                                      [Arabic]
Finland                                   [Finnish]
France                                     [French]
Georgia                                  [Georgian]
Germany                                    [German]
Greece                                      [Greek]
Hong Kong (S.A.R)             [Traditional Chinese]
Hungary                                 [Hungarian]
Iceland                                 [Icelandic]
India                                     [Marathi]
Indonesia                              [Indonesian]


Goals
* For each distinct language, what is the lexical ellaboration for each of the 20 emotions?
* For each distinct language, is there any statistical significance in the distribution of the results of any particular emotion?

In [298]:
# drop the rest of the ~75 columns
filtered = filtered[['countryname', 'countryname', 'Language_Name', 'language', 'reason', 'percent', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']].copy()

In [299]:
emotions = ['admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']

In [ ]:
len(emotions)

20

In [300]:
# get the dataset of the dictionaries. We will look at how elaborated each of the twenty emotions are in each of the 39 languages
bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)
bila_nouns_full

/tmp/ipykernel_2586964/268126284.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)


,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,abdomen,2.0,1.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-9.130648,-0.030722,10499,0.000095,0.693147
1,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
2,chi.14718491,absence,4.0,2.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-0.677996,10499,0.000190,1.098612
3,chi.14718491,abstinence,2.0,5.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.031819,3.864680,10499,0.000476,1.791759
4,chi.14718491,abundance,3.0,4.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.214194,0.484280,10499,0.000381,1.609438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824416,wu.89119131142,zephyr,2.0,2.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.530683,1.086020,33359,0.000060,1.098612
1824417,wu.89119131142,zinc,2.0,2.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.530683,-0.009188,33359,0.000060,1.098612
1824418,wu.89119131142,zone,6.0,4.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.019809,-0.305218,33359,0.000120,1.609438
1824419,wu.89119131142,zoo,1.0,1.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.936173,-1.879701,33359,0.000030,0.693147


In [301]:
from pathlib import Path

with open(Path("~/Projects/hypocognition/data/external/stopwords.txt").expanduser()) as f:
    words = [line.strip() for line in f if line.strip()]
words

['word',
 'name',
 'form',
 'verb',
 'sound',
 'noun',
 'letter',
 'language',
 'class',
 'case',
 'mark',
 'comp',
 'note',
 'term',
 'meaning',
 'root',
 'sense',
 'prep',
 'speech',
 'subject',
 'sign',
 'character',
 'prop',
 'dictionary',
 'article',
 'adjective',
 'stem',
 'sentence',
 'particle',
 'dial',
 'person',
 'fig',
 'phrase',
 'section',
 'compound',
 'par',
 'con',
 'sub',
 'thing',
 'place',
 'part',
 'kind',
 'act',
 'way',
 'cause',
 'set',
 'side',
 'piece',
 'end',
 'state',
 'use',
 'round',
 'point',
 'manner',
 'object',
 'change',
 'matter',
 'action',
 'self',
 'top',
 'cover',
 'measure',
 'bit',
 'sort',
 'type',
 'good',
 'colour',
 'lot',
 'min',
 'middle',
 'degree',
 'member',
 'house',
 'home',
 'school',
 'work',
 'life',
 'mind',
 'number',
 'art',
 'world',
 'level',
 'style',
 'variety',
 'amount',
 'show',
 'specie',
 'man',
 'men',
 'woman',
 'women',
 'people',
 'family',
 'father',
 'mother',
 'son',
 'daughter',
 'child',
 'wife',
 'husband',


In [302]:
# remove stop words
bila_nouns_full = bila_nouns_full[~bila_nouns_full["word"].isin(words)]
bila_nouns_full

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,abdomen,2.0,1.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-9.130648,-0.030722,10499,0.000095,0.693147
1,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
2,chi.14718491,absence,4.0,2.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-0.677996,10499,0.000190,1.098612
3,chi.14718491,abstinence,2.0,5.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.031819,3.864680,10499,0.000476,1.791759
4,chi.14718491,abundance,3.0,4.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.214194,0.484280,10499,0.000381,1.609438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824416,wu.89119131142,zephyr,2.0,2.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.530683,1.086020,33359,0.000060,1.098612
1824417,wu.89119131142,zinc,2.0,2.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.530683,-0.009188,33359,0.000060,1.098612
1824418,wu.89119131142,zone,6.0,4.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.019809,-0.305218,33359,0.000120,1.609438
1824419,wu.89119131142,zoo,1.0,1.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.936173,-1.879701,33359,0.000030,0.693147


In [303]:
tot_words = bila_nouns_full.groupby('id')['word'].nunique().reset_index(name='Total_words_in_dict')
tot_words

,id,Total_words_in_dict
0,chi.14718491,1622
1,coo.31924067983704,6458
2,coo.31924099174934,3921
3,coo1.ark:/13960/t9m33d507,4646
4,dictionaria.daakaka_vonprince_daka1243,725
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,2190
612,webonary.tampulma_asare_tamp1252,774
613,webonary.turkmen_almamedow_turk1304,3804
614,wu.89017649658,2842


In [304]:
tot_counts = bila_nouns_full.groupby('id')['count'].sum().reset_index(name='Total_counts_in_dict')
tot_counts

,id,Total_counts_in_dict
0,chi.14718491,10499.0
1,coo.31924067983704,121674.0
2,coo.31924099174934,10358.0
3,coo1.ark:/13960/t9m33d507,20574.0
4,dictionaria.daakaka_vonprince_daka1243,1825.0
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,32836.0
612,webonary.tampulma_asare_tamp1252,4117.0
613,webonary.turkmen_almamedow_turk1304,27273.0
614,wu.89017649658,12018.0


In [305]:
# I think we will needs these later
dictionary_means = (
    bila_nouns_full
        .groupby('id', as_index=False)['count']
        .mean()
        .rename(columns={'count': 'dictionary_count_mean'})
)
dictionary_means

,id,dictionary_count_mean
0,chi.14718491,6.472873
1,coo.31924067983704,18.840818
2,coo.31924099174934,2.641673
3,coo1.ark:/13960/t9m33d507,4.428325
4,dictionaria.daakaka_vonprince_daka1243,2.517241
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,14.993607
612,webonary.tampulma_asare_tamp1252,5.319121
613,webonary.turkmen_almamedow_turk1304,7.169558
614,wu.89017649658,4.228712


In [306]:
bila_nouns_full_emotions = bila_nouns_full[bila_nouns_full['word'].isin(emotions)].reset_index(drop=True)
bila_nouns_full_emotions

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,anger,5.0,2.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-2.335062,10499,0.000190,1.098612
1,chi.14718491,anxiety,2.0,1.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-9.130648,-1.043924,10499,0.000095,0.693147
2,chi.14718491,confusion,5.0,1.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-9.130648,-1.619347,10499,0.000095,0.693147
3,chi.14718491,disgust,3.0,4.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.214194,1.164688,10499,0.000381,1.609438
4,chi.14718491,fear,8.0,9.0,Nancowry,nanc1247,1884,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-1.337584,10499,0.000857,2.302585
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6466,wu.89119131142,love,10.0,75.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-6.296794,3.792428,33359,0.002248,4.330733
6467,wu.89119131142,pleasure,5.0,13.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-7.989972,-2.243187,33359,0.000390,2.639057
6468,wu.89119131142,regret,5.0,11.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-8.144171,1.038386,33359,0.000330,2.484907
6469,wu.89119131142,relief,11.0,10.0,Herero,here1253,1989,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-8.231207,0.866433,33359,0.000300,2.397895


In [307]:
ne = bila_nouns_full_emotions['word'].unique()
ne

array(['anger', 'anxiety', 'confusion', 'disgust', 'fear', 'love',
       'pleasure', 'regret', 'relief', 'admiration', 'boredom',
       'compassion', 'determination', 'frustration', 'gratitude', 'hope',
       'loneliness', 'sadness'], dtype=object)

In [308]:
set(emotions) - set(ne)

{'calm', 'moved'}

So the dataset is missing two of the survey emotions

In [309]:
# already filtered out everything but the 20 emotions
# now we want to filter evrything but the 39 languages
# Where are lang_name and Language_Names the Same?
emotions_in_survey_languages_in_bila = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(filtered['Language_Name'].unique())].reset_index(drop=True)
emotions_in_survey_languages_in_bila

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,coo.31924067983704,admiration,3.0,10.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.371276,-0.854986,121674,0.000082,2.397895
1,coo.31924067983704,anger,5.0,85.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.314243,0.579516,121674,0.000699,4.454347
2,coo.31924067983704,anxiety,2.0,77.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.411944,8.633841,121674,0.000633,4.356709
3,coo.31924067983704,boredom,1.0,11.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.284257,2.582914,121674,0.000090,2.484907
4,coo.31924067983704,compassion,2.0,33.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-8.242632,3.250008,121674,0.000271,3.526361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,uva.x004877953,love,10.0,639.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-6.115647,15.519650,282895,0.002259,6.461468
428,uva.x004877953,pleasure,5.0,286.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-6.918850,7.953903,282895,0.001011,5.659482
429,uva.x004877953,regret,5.0,107.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-7.896818,5.664047,282895,0.000378,4.682131
430,uva.x004877953,relief,11.0,89.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-8.079201,3.907437,282895,0.000315,4.499810


In [310]:
# I asked gemini to map our study languages names to language names from the bila dataset
covid_to_bila_nouns_full_lang_name_mapping = pd.read_csv(Path("~/Projects/hypocognition/data/external/covid_to_full_bila_lang_name_mapping.csv"))
covid_to_bila_nouns_full_lang_name_mapping

,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,EN,English,NaN,"Midland American English, Singlish, Devon, Sus..."
2,ID,Indonesian,Indonesian,"Standard Malay, Central Malay, Baba Malay"
3,ZH-T,Traditional Chinese,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
4,VI,Vietnamese,Vietnamese,Nung (Viet Nam)
5,DA,Danish,Danish,NaN
6,AR,Arabic,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar..."
7,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
8,AFRI,Afrikaans,Afrikaans,Dutch
9,ES-ES,Spanish (Spain),Spanish,NaN


In [311]:
# Update selection table 
selection_in_bila =selection.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on="study_language_names",
    how="left"
)

selection_in_bila

,countryname,language,lang_responses,total_responses,percent,Language_Name,Selected,reason,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,0,We will leave out Australia for now because i...,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Brazil,PT-BR,306,325,94.153846,Brazilian Portuguese,1,Let's use Brazilian Portuguese for Brazil,PT-BR,Brazilian Portuguese,Brazilian Portuguese,Portuguese
2,Bulgaria,BG,269,276,97.463768,Bulgarian,1,Let's use Bulgarian for Bulgaria,BG,Bulgarian,Bulgarian,NaN
3,Canada,EN,195,286,68.181818,English,0,"Doesn't look good for Canada, we will drop the...",EN,English,NaN,"Midland American English, Singlish, Devon, Sus..."
4,Chile,ES,469,493,95.131846,Spanish,1,Let's use Spanish (General) for chile,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
5,China,ZH-S,1988,2009,98.954704,Simplified Chinese,1,Let's use Simplified Chinese for china,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
6,Colombia,ES,122,237,51.476793,Spanish,1,Let's use Spanish (General) for Colombia,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
7,Croatia,HR,524,528,99.242424,Croatian,1,Let's use croatian for croatia,HR,Croatian,Serbian-Croatian-Bosnian,NaN
8,Curacao,NL,180,235,76.595745,Dutch,1,Let's use Dutch for Curacao,NL,Dutch,Dutch,Western Flemish
9,Denmark,DA,148,233,63.519313,Danish,1,Let's use Danish for Denmark,DA,Danish,Danish,NaN


In [313]:
selection_in_bila.columns

Index(['countryname', 'language', 'lang_responses', 'total_responses',
       'percent', 'Language_Name', 'Selected', 'reason', 'code',
       'study_language_names', 'bila_language_name_mapping',
       'possible_alternative_bila_language_name_mappings'],
      dtype='object')

In [314]:
selection_in_bila = selection_in_bila[['countryname', 'language','study_language_names', 'bila_language_name_mapping', 'lang_responses', 'total_responses',
       'percent', 'Selected', 'reason',
       'possible_alternative_bila_language_name_mappings']]
selection_in_bila

,countryname,language,study_language_names,bila_language_name_mapping,lang_responses,total_responses,percent,Selected,reason,possible_alternative_bila_language_name_mappings
0,Australia,ZH-S,Simplified Chinese,Mandarin Chinese,181,378,47.883598,0,We will leave out Australia for now because i...,"Beijing Mandarin, Wu Chinese"
1,Brazil,PT-BR,Brazilian Portuguese,Brazilian Portuguese,306,325,94.153846,1,Let's use Brazilian Portuguese for Brazil,Portuguese
2,Bulgaria,BG,Bulgarian,Bulgarian,269,276,97.463768,1,Let's use Bulgarian for Bulgaria,NaN
3,Canada,EN,English,NaN,195,286,68.181818,0,"Doesn't look good for Canada, we will drop the...","Midland American English, Singlish, Devon, Sus..."
4,Chile,ES,Spanish,Spanish,469,493,95.131846,1,Let's use Spanish (General) for chile,"Latin American Spanish, Mexican Spanish"
5,China,ZH-S,Simplified Chinese,Mandarin Chinese,1988,2009,98.954704,1,Let's use Simplified Chinese for china,"Beijing Mandarin, Wu Chinese"
6,Colombia,ES,Spanish,Spanish,122,237,51.476793,1,Let's use Spanish (General) for Colombia,"Latin American Spanish, Mexican Spanish"
7,Croatia,HR,Croatian,Serbian-Croatian-Bosnian,524,528,99.242424,1,Let's use croatian for croatia,NaN
8,Curacao,NL,Dutch,Dutch,180,235,76.595745,1,Let's use Dutch for Curacao,Western Flemish
9,Denmark,DA,Danish,Danish,148,233,63.519313,1,Let's use Danish for Denmark,NaN


In [315]:
selection_in_bila.to_csv("~/Projects/hypocognition/data/processed/selection_in_bila.csv")

In [316]:
# Add a language name pa for the the bila dataset language names
filtered_mapped =filtered.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on="study_language_names",
    how="left"
)

filtered_mapped

,countryname,countryname,Language_Name,language,reason,percent,admiration,calm,compassion,determination,...,disgust,fear,frustration,loneliness,regret,sadness,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,1.0,6.0,1.0,0.0,...,6.0,6.0,6.0,6.0,6.0,6.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
1,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,4.0,4.0,2.0,3.0,...,2.0,2.0,2.0,4.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
2,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,3.0,3.0,3.0,3.0,...,0.0,3.0,0.0,3.0,1.0,4.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
3,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,6.0,3.0,6.0,4.0,...,0.0,3.0,4.0,2.0,0.0,2.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
4,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,5.0,5.0,5.0,6.0,...,2.0,2.0,3.0,0.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15017,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,1.0,3.0,3.0,1.0,...,2.0,3.0,2.0,3.0,3.0,2.0,RU,Russian,Russian,Belarusian
15018,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,6.0,2.0,...,1.0,5.0,2.0,2.0,6.0,5.0,RU,Russian,Russian,Belarusian
15019,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,3.0,3.0,...,3.0,2.0,5.0,4.0,4.0,3.0,RU,Russian,Russian,Belarusian
15020,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,2.0,3.0,6.0,2.0,...,3.0,3.0,3.0,1.0,6.0,4.0,RU,Russian,Russian,Belarusian


In [317]:
filtered_mapped = filtered_mapped[filtered_mapped['bila_language_name_mapping'].notna()]
filtered_mapped

,countryname,countryname,Language_Name,language,reason,percent,admiration,calm,compassion,determination,...,disgust,fear,frustration,loneliness,regret,sadness,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,1.0,6.0,1.0,0.0,...,6.0,6.0,6.0,6.0,6.0,6.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
1,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,4.0,4.0,2.0,3.0,...,2.0,2.0,2.0,4.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
2,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,3.0,3.0,3.0,3.0,...,0.0,3.0,0.0,3.0,1.0,4.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
3,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,6.0,3.0,6.0,4.0,...,0.0,3.0,4.0,2.0,0.0,2.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
4,Malaysia,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,5.0,5.0,5.0,6.0,...,2.0,2.0,3.0,0.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15017,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,1.0,3.0,3.0,1.0,...,2.0,3.0,2.0,3.0,3.0,2.0,RU,Russian,Russian,Belarusian
15018,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,6.0,2.0,...,1.0,5.0,2.0,2.0,6.0,5.0,RU,Russian,Russian,Belarusian
15019,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,3.0,3.0,...,3.0,2.0,5.0,4.0,4.0,3.0,RU,Russian,Russian,Belarusian
15020,Kazakhstan,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,2.0,3.0,6.0,2.0,...,3.0,3.0,3.0,1.0,6.0,4.0,RU,Russian,Russian,Belarusian


In [318]:
filtered_mapped["study_language_names"].unique()

array(['Malay', 'Dutch', 'Spanish (Spain)', 'Russian', 'Serbian',
       'Afrikaans', 'Swedish', 'Arabic', 'Traditional Chinese (Taiwan)',
       'Turkish', 'Ukrainian', 'Vietnamese', 'Brazilian Portuguese',
       'Bulgarian', 'Spanish', 'Simplified Chinese', 'Croatian', 'Danish',
       'Finnish', 'French', 'Georgian', 'German', 'Greek',
       'Traditional Chinese', 'Hungarian', 'Icelandic', 'Marathi',
       'Indonesian', 'Persian', 'Hebrew', 'Italian', 'Japanese'],
      dtype=object)

In [319]:
filtered_mapped["study_language_names"].nunique()

32

In [320]:
filtered_mapped["bila_language_name_mapping"].nunique()

28

In [321]:
filtered_mapped['countryname'].nunique()

countryname    38
countryname    38
dtype: int64

In [322]:
set(lang_names["Language_Name"]) - set(selection[selection['Selected']==1]['Language_Name'])

{'Catalan',
 'Czech',
 'Dari',
 'English',
 'Estonian',
 'Filipino',
 'Hindi',
 'Kazakh',
 'Korean',
 'Malayalam',
 'Norwegian',
 'Polish',
 'Portuguese',
 'Romanian',
 'Slovak',
 'Slovenian'}

In [323]:
set(selection[selection['Selected']==1]['Language_Name']) - set(selection_in_bila[selection_in_bila['bila_language_name_mapping'].notna()]['study_language_names'])

{'Maltese'}

In [324]:
filtered_mapped = filtered_mapped.loc[:, ~filtered_mapped.columns.duplicated()]

In [325]:
languages_per_country[languages_per_country['countryname'] =="Norway"]

,countryname,language,lang_responses,total_responses,percent,Language_Name


## the discrepency is because Mandarin Chinese, [No Direct Match], Serbian-Croatian-Bosnian, and something else appear for more than one country. 39 countries -> 33 response languages -> 30 bila dictionarys.

In [326]:
filtered_mapped

,countryname,Language_Name,language,reason,percent,admiration,calm,compassion,determination,moved,...,disgust,fear,frustration,loneliness,regret,sadness,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,1.0,6.0,1.0,0.0,0.0,...,6.0,6.0,6.0,6.0,6.0,6.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
1,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,4.0,4.0,2.0,3.0,2.0,...,2.0,2.0,2.0,4.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
2,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,3.0,3.0,3.0,3.0,3.0,...,0.0,3.0,0.0,3.0,1.0,4.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
3,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,6.0,3.0,6.0,4.0,6.0,...,0.0,3.0,4.0,2.0,0.0,2.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
4,Malaysia,Malay,MS,Let's use Malay for Malaysia,72.429907,5.0,5.0,5.0,6.0,4.0,...,2.0,2.0,3.0,0.0,2.0,3.0,MS,Malay,Standard Malay,"Central Malay, Baba Malay, Malayo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15017,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,1.0,3.0,3.0,1.0,5.0,...,2.0,3.0,2.0,3.0,3.0,2.0,RU,Russian,Russian,Belarusian
15018,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,6.0,2.0,6.0,...,1.0,5.0,2.0,2.0,6.0,5.0,RU,Russian,Russian,Belarusian
15019,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,3.0,3.0,3.0,3.0,4.0,...,3.0,2.0,5.0,4.0,4.0,3.0,RU,Russian,Russian,Belarusian
15020,Kazakhstan,Russian,RU,Let's use Russian for Kazakhstan,95.876289,2.0,3.0,6.0,2.0,2.0,...,3.0,3.0,3.0,1.0,6.0,4.0,RU,Russian,Russian,Belarusian


In [327]:
bila_nouns_full_emotions_filtered = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(covid_to_bila_nouns_full_lang_name_mapping['bila_language_name_mapping'].values)]
bila_nouns_full_emotions_filtered

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
9,coo.31924067983704,admiration,3.0,10.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.371276,-0.854986,121674,0.000082,2.397895
10,coo.31924067983704,anger,5.0,85.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.314243,0.579516,121674,0.000699,4.454347
11,coo.31924067983704,anxiety,2.0,77.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.411944,8.633841,121674,0.000633,4.356709
12,coo.31924067983704,boredom,1.0,11.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.284257,2.582914,121674,0.000090,2.484907
13,coo.31924067983704,compassion,2.0,33.0,Turkish,nucl1301,1991,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-8.242632,3.250008,121674,0.000271,3.526361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6145,uva.x004877953,love,10.0,639.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-6.115647,15.519650,282895,0.002259,6.461468
6146,uva.x004877953,pleasure,5.0,286.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-6.918850,7.953903,282895,0.001011,5.659482
6147,uva.x004877953,regret,5.0,107.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-7.896818,5.664047,282895,0.000378,4.682131
6148,uva.x004877953,relief,11.0,89.0,Japanese,nucl1643,1974,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-8.079201,3.907437,282895,0.000315,4.499810


In [337]:
bila_nouns_full_emotions_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 9 to 6149
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      713 non-null    object 
 1   word                    713 non-null    object 
 2   nsenses                 713 non-null    float64
 3   count                   713 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    int64  
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    float6

In [343]:
# add the rows of the emotions words which don't appear in each dictionary
bila_emotions = bila_nouns_full_emotions_filtered['word'].unique()
full_index = pd.MultiIndex.from_product(
    [bila_nouns_full_emotions_filtered["id"].unique(), bila_emotions],
    names=["id", "word"]
)


In [344]:

bila_nouns_full_emotions_filtered_full = (
    bila_nouns_full_emotions_filtered
    .set_index(["id", "word"])
    .reindex(full_index)
    .reset_index()
)
num_cols = ["nsenses", "count", "log_count"]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 713 non-null    float64
 3   count                   713 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    float64
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    fl

In [345]:

#set the number columns to 0 where NaN
bila_nouns_full_emotions_filtered_full[num_cols] = bila_nouns_full_emotions_filtered_full[num_cols].fillna(0)
meta_cols = [
    "langname", "glottocode", "year", "title", "imprint", "author",
    "area", "langfamily", "affiliation", "longitude", "latitude"
]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 738 non-null    float64
 3   count                   738 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    float64
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    fl

In [346]:
# fill the those zero rows with dictionary meta -data
bila_nouns_full_emotions_filtered_full[meta_cols] = (
    bila_nouns_full_emotions_filtered_full
    .groupby("id")[meta_cols]
    .transform("first")
)

bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 738 non-null    float64
 3   count                   738 non-null    float64
 4   langname                738 non-null    object 
 5   glottocode              738 non-null    object 
 6   year                    738 non-null    float64
 7   title                   738 non-null    object 
 8   imprint                 738 non-null    object 
 9   author                  738 non-null    object 
 10  area                    738 non-null    object 
 11  langfamily              738 non-null    object 
 12  affiliation             738 non-null    object 
 13  longitude               738 non-null    float64
 14  latitude                738 non-null    fl

In [347]:
bila_nouns_full_emotions_filtered_full[bila_nouns_full_emotions_filtered_full['regression_elaboration'].isna()]

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
27,inu.30000042071369,frustration,0.0,0.0,Slovenian,slov1268,1994.0,"Angleško-slovenski, slovensko-angleški slovar ...","Cankarjeva zal., 1994.",0,Eurasia,Indo-European,"Indo-European, Classical Indo-European, Balto-...",14.776600,46.254300,NaN,NaN,NaN,NaN,0.0
47,inu.30000078219932,hope,0.0,0.0,Estonian,esto1258,1999.0,Eesti-Inglise sõnaraamat = Estonian-English Di...,"Festart, 1999.",0,Eurasia,Uralic,"Uralic, Finnic, Coastal Finnic, Neva, Central ...",25.820000,58.550000,NaN,NaN,NaN,NaN,0.0
83,inu.30000083684336,hope,0.0,0.0,Slovak,slov1269,2002.0,Slovensko-anglický slovník = Slovak-English di...,"Ikar, 2002.",0,Eurasia,Indo-European,"Indo-European, Classical Indo-European, Balto-...",18.784790,48.545705,NaN,NaN,NaN,NaN,0.0
153,mdp.39015015059622,frustration,0.0,0.0,Polish,poli1260,1962.0,The Kościuszko Foundation dictionary: English-...,"Kosciuszko Foundation, 1960-1962.",0,Eurasia,Indo-European,"Indo-European, Classical Indo-European, Balto-...",18.625500,51.843900,NaN,NaN,NaN,NaN,0.0
198,mdp.39015041370621,admiration,0.0,0.0,Kazakh,kaza1248,1994.0,Kazakh (Qazaq)-English dictionary / Karl A. Kr...,"Dunwoody Press, c1994.",0,Eurasia,Turkic,"Turkic, Common Turkic, Kipchak, South Kipchak",71.454000,51.170000,NaN,NaN,NaN,NaN,0.0
204,mdp.39015041370621,determination,0.0,0.0,Kazakh,kaza1248,1994.0,Kazakh (Qazaq)-English dictionary / Karl A. Kr...,"Dunwoody Press, c1994.",0,Eurasia,Turkic,"Turkic, Common Turkic, Kipchak, South Kipchak",71.454000,51.170000,NaN,NaN,NaN,NaN,0.0
207,mdp.39015041370621,frustration,0.0,0.0,Kazakh,kaza1248,1994.0,Kazakh (Qazaq)-English dictionary / Karl A. Kr...,"Dunwoody Press, c1994.",0,Eurasia,Turkic,"Turkic, Common Turkic, Kipchak, South Kipchak",71.454000,51.170000,NaN,NaN,NaN,NaN,0.0
333,mdp.39015055371309,frustration,0.0,0.0,Marathi,mara1378,1965.0,The popular modern dictionary (English-English...,"Educational Pub. Co., 1965.",0,Eurasia,Indo-European,"Indo-European, Classical Indo-European, Indo-I...",76.666500,17.934400,NaN,NaN,NaN,NaN,0.0
345,mdp.39015058560494,boredom,0.0,0.0,Bulgarian,bulg1262,1914.0,Complete Bulgarian-English dictionary,"J. H. Nickoloff, 1914.",0,Eurasia,Indo-European,"Indo-European, Classical Indo-European, Balto-...",25.047000,43.364600,NaN,NaN,NaN,NaN,0.0
449,miun.aee0517.0001.001,sadness,0.0,0.0,Hungarian,hung1274,1924.0,A dictionary of the Hungarian and English lang...,"Franklin-társulat, 1908-24.",0,Eurasia,Uralic,"Uralic, Hungaric",19.655527,46.906859,NaN,NaN,NaN,NaN,0.0


In [ ]:
for i in  bila_nouns_full_emotions_filtered[]

Index(['id', 'word', 'nsenses', 'count', 'langname', 'glottocode', 'year',
       'title', 'imprint', 'author', 'area', 'langfamily', 'affiliation',
       'longitude', 'latitude', 'estimate', 'regression_elaboration',
       'dictsize_data', 'simple_elaboration', 'log_count'],
      dtype='object')

In [330]:
elab_per_emotion = bila_nouns_full_emotions_filtered_full[['langname',   'glottocode','word', 'count',  'id', 'year',"simple_elaboration", "regression_elaboration",	"dictsize_data"	]]
elab_per_emotion

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data
0,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0
1,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0
2,Turkish,nucl1301,anxiety,77.0,coo.31924067983704,1991.0,0.000633,8.633841,121674.0
3,Turkish,nucl1301,boredom,11.0,coo.31924067983704,1991.0,0.000090,2.582914,121674.0
4,Turkish,nucl1301,compassion,33.0,coo.31924067983704,1991.0,0.000271,3.250008,121674.0
...,...,...,...,...,...,...,...,...,...
733,Japanese,nucl1643,love,639.0,uva.x004877953,1974.0,0.002259,15.519650,282895.0
734,Japanese,nucl1643,pleasure,286.0,uva.x004877953,1974.0,0.001011,7.953903,282895.0
735,Japanese,nucl1643,regret,107.0,uva.x004877953,1974.0,0.000378,5.664047,282895.0
736,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0


In [331]:
elab_per_emotion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   langname                738 non-null    object 
 1   glottocode              738 non-null    object 
 2   word                    738 non-null    object 
 3   count                   738 non-null    float64
 4   id                      738 non-null    object 
 5   year                    738 non-null    float64
 6   simple_elaboration      713 non-null    float64
 7   regression_elaboration  713 non-null    float64
 8   dictsize_data           713 non-null    float64
dtypes: float64(5), object(4)
memory usage: 52.0+ KB


In [ ]:
stats_by_country = (
    filtered_mapped
    .groupby(['countryname', 'bila_language_name_mapping'])[emotions]
    .agg(['mean', 'std'])
)
stats_by_country = pd.DataFrame(stats_by_country)
stats_by_country

admiration                calm  \
                                                   mean       std      mean   
countryname       bila_language_name_mapping                                  
Brazil            Brazilian Portuguese         3.540984  1.745047  3.421569   
Bulgaria          Bulgarian                    2.498141  1.980548  3.334572   
Chile             Spanish                      2.867521  1.901080  2.899358   
China             Mandarin Chinese             4.365940  1.736698  3.914807   
Colombia          Spanish                      3.409836  1.974022  3.418033   
Croatia           Serbian-Croatian-Bosnian     2.552581  1.680006  3.332061   
Curacao           Dutch                        3.627778  1.743401  3.961111   
Denmark           Danish                       2.668919  1.513605  3.445946   
Egypt             Arabic                       2.547472  1.886575  3.622222   
Finland           Finnish                      2.738462  1.547467  3.726923   
France            French                       3.357977  1.801764  3.746094   
Georgia           Georgian                     2.523605  1.866354  3.248927   
Germany           German                       2.804124  1.800225  3.628866   
Greece            Greek                        2.328889  1.887059  3.337778   
Hong Kong (S.A.R) Mandarin Chinese             2.669065  1.770894  3.064748   
Hungary           Hungarian                    2.679688  1.925815  3.265625   
Iceland           Icelandic                    3.469741  1.867297  3.299712   
India             Marathi                      3.500000  1.768410  3.982143   
Indonesia         Indonesian                   3.201863  1.577327  3.834365   
Iran              Persian                      2.347826  1.745782  2.774194   
Israel            Hebrew                       1.569536  1.718565  3.240000   
Italy             Italian                      2.966981  1.900316  3.327059   
Japan             Japanese                     2.602176  1.523357  3.559441   
Jordan            Arabic                       3.108225  1.965235  3.952586   
Kazakhstan        Russian                      2.438849  1.958468  3.469534   
Malaysia          Standard Malay               3.793548  1.548761  4.000000   
Netherlands       Dutch                        3.325048  1.646279  3.671756   
Peru              Spanish                      3.706161  1.884501  3.693023   
Russia            Russian                      2.553043  1.774840  3.349565   
Serbia            Serbian-Croatian-Bosnian     2.712644  1.846614  3.517143   
South Africa      Afrikaans                    2.946429  1.712904  3.642857   
Spain             Spanish                      3.428571  1.994498  3.428571   
Sweden            Swedish                      3.300000  1.774224  3.580000   
Syria             Arabic                       1.849372  1.996402  3.377593   
Taiwan            Mandarin Chinese             3.993307  1.660520  3.666667   
Turkey            Turkish                      2.027888  1.760460  3.611111   
Ukraine           Ukrainian                    2.856489  1.644428  3.203957   
Vietnam           Vietnamese                   4.219178  1.746841  4.263699   

                                                       compassion            \
                                                   std       mean       std   
countryname       bila_language_name_mapping                                  
Brazil            Brazilian Portuguese        1.502585   4.633987  1.319471   
Bulgaria          Bulgarian                   1.712431   4.313433  1.590788   
Chile             Spanish                     1.546330   3.893390  1.693798   
China             Mandarin Chinese            1.683731   4.240976  1.701800   
Colombia          Spanish                     1.419115   4.024590  1.678563   
Croatia           Serbian-Croatian-Bosnian    1.485572   2.904398  1.650043   
Curacao           Dutch                       1.607816   4.777778  1.174951   
Denmark        

In [ ]:
stats_by_country.columns = [
    f"{emotion}_{stat}" for emotion, stat in stats_by_country.columns
]
stats_by_country = stats_by_country.reset_index()
stats_by_country

,countryname,bila_language_name_mapping,admiration_mean,admiration_std,calm_mean,calm_std,compassion_mean,compassion_std,determination_mean,determination_std,...,fear_mean,fear_std,frustration_mean,frustration_std,loneliness_mean,loneliness_std,regret_mean,regret_std,sadness_mean,sadness_std
0,Brazil,Brazilian Portuguese,3.540984,1.745047,3.421569,1.502585,4.633987,1.319471,3.725490,1.593986,...,3.603279,1.757500,3.437908,1.941201,2.518033,2.069635,1.934426,1.881951,3.321311,1.833991
1,Bulgaria,Bulgarian,2.498141,1.980548,3.334572,1.712431,4.313433,1.590788,3.565056,1.684113,...,2.561338,1.954938,3.029740,2.089207,2.334572,2.154385,2.929104,2.009018,3.252788,1.930519
2,Chile,Spanish,2.867521,1.901080,2.899358,1.546330,3.893390,1.693798,3.370450,1.700109,...,3.528785,1.836746,4.119914,1.731608,2.869658,2.067930,2.352564,1.969156,3.799145,1.745670
3,China,Mandarin Chinese,4.365940,1.736698,3.914807,1.683731,4.240976,1.701800,4.165992,1.693406,...,1.809500,1.726083,1.815642,1.763461,1.812437,1.826492,1.477009,1.701083,2.256709,1.848471
4,Colombia,Spanish,3.409836,1.974022,3.418033,1.419115,4.024590,1.678563,3.818182,1.653280,...,3.024590,1.943258,3.409836,1.897167,2.578512,2.170837,1.721311,1.601973,3.459016,1.916210
5,Croatia,Serbian-Croatian-Bosnian,2.552581,1.680006,3.332061,1.485572,2.904398,1.650043,3.508604,1.438863,...,2.739962,1.825025,3.430210,1.808983,2.692748,1.952364,2.795802,1.713266,2.990458,1.759406
6,Curacao,Dutch,3.627778,1.743401,3.961111,1.607816,4.777778,1.174951,4.000000,1.549914,...,2.477778,1.933115,3.005556,1.877527,1.894444,1.853570,1.350000,1.645970,2.622222,1.906049
7,Denmark,Danish,2.668919,1.513605,3.445946,1.531012,4.067568,1.450602,3.351351,1.483996,...,1.824324,1.610832,3.148649,1.852980,2.277027,2.026427,1.479730,1.655622,1.601351,1.732993
8,Egypt,Arabic,2.547472,1.886575,3.622222,1.728264,4.173006,1.710858,3.161090,1.728845,...,3.556790,1.953694,3.419434,2.007981,3.686275,2.140949,2.710591,2.129295,4.047970,1.806232
9,Finland,Finnish,2.738462,1.547467,3.726923,1.380520,4.042471,1.350471,3.532819,1.327221,...,2.509653,1.600298,3.846154,1.699246,2.776062,1.927978,1.534615,1.570343,2.953668,1.672212


In [ ]:
stats_long = (
    stats_by_country
    .set_index(['countryname', 'bila_language_name_mapping'])
    .filter(regex='_(mean|std)$')
    .stack()
    .reset_index()
)

stats_long[['word', 'stat']] = stats_long['level_2'].str.rsplit('_', n=1, expand=True)
stats_long = stats_long.rename(columns={0: 'value'}).drop(columns='level_2')

stats_long = (
    stats_long
    .pivot_table(
        index=['countryname', 'bila_language_name_mapping', 'word'],
        columns='stat',
        values='value'
    )
    .reset_index()
)


,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std
0,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460
1,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473
2,Turkish,nucl1301,anxiety,77.0,coo.31924067983704,1991.0,0.000633,8.633841,121674.0,Turkey,Turkish,3.150794,1.862045
3,Turkish,nucl1301,boredom,11.0,coo.31924067983704,1991.0,0.000090,2.582914,121674.0,Turkey,Turkish,3.529644,1.880379
4,Turkish,nucl1301,compassion,33.0,coo.31924067983704,1991.0,0.000271,3.250008,121674.0,Turkey,Turkish,3.956349,1.627111
...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,Japanese,nucl1643,love,639.0,uva.x004877953,1974.0,0.002259,15.519650,282895.0,Japan,Japanese,3.614130,1.511732
914,Japanese,nucl1643,pleasure,286.0,uva.x004877953,1974.0,0.001011,7.953903,282895.0,Japan,Japanese,2.717172,1.495210
915,Japanese,nucl1643,regret,107.0,uva.x004877953,1974.0,0.000378,5.664047,282895.0,Japan,Japanese,1.956522,1.623824
916,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428


In [ ]:

merged = elab_per_emotion.merge(
    stats_long,
    left_on=['langname', 'word'],
    right_on=['bila_language_name_mapping', 'word'],
    how='left'
)

merged

In [178]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    684 non-null    object 
 1   glottocode                  684 non-null    object 
 2   word                        684 non-null    object 
 3   count                       684 non-null    float64
 4   id                          684 non-null    object 
 5   year                        684 non-null    float64
 6   simple_elaboration          666 non-null    float64
 7   regression_elaboration      666 non-null    float64
 8   dictsize_data               666 non-null    float64
 9   countryname                 684 non-null    object 
 10  bila_language_name_mapping  684 non-null    object 
 11  response_mean               684 non-null    float64
 12  response_std                684 non-null    float64
 13  dictionary_count_mean       684 non

In [169]:
# add the dictionary_count_mean
merged = merged.merge(
    dictionary_means,
    left_on='id',
    right_on='id',
    how='left'
)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std,dictionary_count_mean
0,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460,18.840818
1,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473,18.840818
2,Turkish,nucl1301,anxiety,77.0,coo.31924067983704,1991.0,0.000633,8.633841,121674.0,Turkey,Turkish,3.150794,1.862045,18.840818
3,Turkish,nucl1301,boredom,11.0,coo.31924067983704,1991.0,0.000090,2.582914,121674.0,Turkey,Turkish,3.529644,1.880379,18.840818
4,Turkish,nucl1301,compassion,33.0,coo.31924067983704,1991.0,0.000271,3.250008,121674.0,Turkey,Turkish,3.956349,1.627111,18.840818
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,Japanese,nucl1643,love,639.0,uva.x004877953,1974.0,0.002259,15.519650,282895.0,Japan,Japanese,3.614130,1.511732,36.592291
914,Japanese,nucl1643,pleasure,286.0,uva.x004877953,1974.0,0.001011,7.953903,282895.0,Japan,Japanese,2.717172,1.495210,36.592291
915,Japanese,nucl1643,regret,107.0,uva.x004877953,1974.0,0.000378,5.664047,282895.0,Japan,Japanese,1.956522,1.623824,36.592291
916,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428,36.592291


In [170]:
merged = merged[merged['countryname'].notna()]

In [2041]:
len(['Turkey', 'Croatia', 'Serbia', 'Malaysia', 'Italy', 'Egypt',
       'Jordan', 'Syria', 'Kazakhstan', 'Russia', 'Chile', 'Colombia',
       'Sweden', 'South Africa', 'India', 'Bulgaria', 'Finland', 'France',
       'Iran', 'Hungary', 'Brazil', 'Vietnam', 'Curacao', 'Netherlands',
       'Germany', 'Indonesia', 'Greece', 'Iceland', 'Denmark', 'Ukraine',
       'China', 'Hong Kong (S.A.R)', 'Taiwan', 'Peru', 'Spain', 'Georgia',
       'Israel', 'Japan'])
len(['Turkey', "nan", 'Croatia', 'Serbia', 'Malaysia', 'Italy', 'Egypt',
       'Jordan', 'Syria', 'Kazakhstan', 'Russia', 'Sweden',
       'South Africa', 'India', 'Bulgaria', 'Finland', 'France', 'Iran',
       'Hungary', 'Brazil', 'Vietnam', 'Curacao', 'Netherlands',
       'Germany', 'Indonesia', 'Greece', 'Iceland', 'Denmark', 'Ukraine',
       'China', 'Hong Kong (S.A.R)', 'Taiwan', 'Chile', 'Colombia',
       'Peru', 'Spain', 'Georgia', 'Israel', 'Japan'])

39

In [2100]:
merged.columns

Index(['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'response_mean',
       'response_std', 'dictionary_count_mean'],
      dtype='object')

In [171]:
# clarify the meaning of mean
merged = merged.rename(columns={'mean': "response_mean"})
# Clarify what std we're talking about
merged = merged.rename(columns={'std': "response_std"})
# drop a bunch of columns

merged = merged[['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'response_mean',
       'response_std', 'dictionary_count_mean' ]]
merged = merged.sort_values(by=["langname", 'word'])
merged


,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
378,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,27.878534
379,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,27.878534
380,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,27.878534
381,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,27.878534
382,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,27.878534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,34.618430
554,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,34.618430
555,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,34.618430
556,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,34.618430


In [172]:
merged = merged.reset_index(drop=True)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,27.878534
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,27.878534
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,27.878534
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,27.878534
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,27.878534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,34.618430
680,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,34.618430
681,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,34.618430
682,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,34.618430


In [2047]:
merged[merged['countryname']=="China"]

,langname,countryname,word,count,dictionary_count_mean,Total_words_in_dict,Total_counts_in_dict,response_mean,response_std,id,year,glottocode
360,Mandarin Chinese,China,admiration,24.0,15.648172,7768,121555.0,4.365940,1.736698,uc1.l0088545033,2004.0,mand1415
363,Mandarin Chinese,China,anger,59.0,15.648172,7768,121555.0,1.956434,1.812030,uc1.l0088545033,2004.0,mand1415
366,Mandarin Chinese,China,anxiety,33.0,15.648172,7768,121555.0,2.632892,1.879226,uc1.l0088545033,2004.0,mand1415
369,Mandarin Chinese,China,boredom,5.0,15.648172,7768,121555.0,2.329782,1.948270,uc1.l0088545033,2004.0,mand1415
372,Mandarin Chinese,China,compassion,9.0,15.648172,7768,121555.0,4.240976,1.701800,uc1.l0088545033,2004.0,mand1415
375,Mandarin Chinese,China,confusion,19.0,15.648172,7768,121555.0,2.270421,1.809292,uc1.l0088545033,2004.0,mand1415
378,Mandarin Chinese,China,determination,8.0,15.648172,7768,121555.0,4.165992,1.693406,uc1.l0088545033,2004.0,mand1415
381,Mandarin Chinese,China,disgust,3.0,15.648172,7768,121555.0,1.745197,1.751591,uc1.l0088545033,2004.0,mand1415
384,Mandarin Chinese,China,fear,89.0,15.648172,7768,121555.0,1.809500,1.726083,uc1.l0088545033,2004.0,mand1415
387,Mandarin Chinese,China,frustration,7.0,15.648172,7768,121555.0,1.815642,1.763461,uc1.l0088545033,2004.0,mand1415


In [173]:
merged['word'].nunique()

18

In [358]:
merged = merged.sort_values(by=["langname", 'countryname','word'])
merged = merged.reset_index(drop=True)
merged = merged.fillna(0)


In [359]:
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,27.878534
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,27.878534
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,27.878534
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,27.878534
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,27.878534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,34.618430
680,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,34.618430
681,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,34.618430
682,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,34.618430


In [360]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    684 non-null    object 
 1   glottocode                  684 non-null    object 
 2   word                        684 non-null    object 
 3   count                       684 non-null    float64
 4   id                          684 non-null    object 
 5   year                        684 non-null    float64
 6   simple_elaboration          684 non-null    float64
 7   regression_elaboration      684 non-null    float64
 8   dictsize_data               684 non-null    float64
 9   countryname                 684 non-null    object 
 10  bila_language_name_mapping  684 non-null    object 
 11  response_mean               684 non-null    float64
 12  response_std                684 non-null    float64
 13  dictionary_count_mean       684 non

In [361]:
merged.to_csv(Path("~/Projects/hypocognition/data/processed/covid_bila_merge.csv").expanduser())

Moving forward, could you create a table with the following columns (broken down into the 36 samples):
The sample country
The language
The correlation between the log count and the response means
The correlation between the log count and the response SDs
The correlation between the log count and the absolute distance of the response means from the scale midpoint

In [ ]:
merged['response_mean'].mean()

np.float64(3.005020991118967)

In [2113]:

# log count (add 1 to avoid log(0) just in case)
merged2 = merged.copy()
merged2["log_count"] = np.log(merged2["count"] + 1)

def corr_within_group(df, x, y):
    return df[x].corr(df[y])

countries = merged2['countryname'].unique().tolist()

for c in countries:
    df = merged2[merged2['countryname']==c]
    print(c,df['log_count'].corr(df['response_mean']), df['log_count'].corr(df['response_std']))





South Africa 0.09148865168961892 -0.12653514046747824
Egypt -0.11433067618319566 0.13329119935570388
Jordan -0.22730182023960746 0.22199851220158942
Syria 0.08928912107744537 -0.29449735181619413
Brazil 0.1624355100367705 -0.09586451876984464
Bulgaria 0.2914576122549996 -0.510940348557617
Denmark -0.3228657980051242 -0.19173518683367818
Curacao -0.22717185923377653 -0.01279205981994988
Netherlands -0.22757861410314467 -0.30170812809299574
Finland 0.0720360849933431 -0.29966805278456204
France -0.12485678545510513 0.16444273794734743
Georgia -0.09763318852656655 0.08255505707391594
Germany 0.17814793850857474 -0.09156428792952183
Greece 0.155470088914526 -0.2718482113675404
Israel -0.19576632870638763 0.22289222905270406
Hungary 0.27087397624337056 -0.3665163766184561
Iceland 0.13037213359602945 -0.3824791026602647
Indonesia -0.01639818458273091 -0.10834449797267735
Italy -0.02900174620842426 -0.3471079511701444
Japan 0.0804512593809085 -0.4519920306637275
China 0.19149418297264034 -0.4

In [2115]:
import numpy as np

merged2 = merged.copy()

# log count
merged2["log_count"] = np.log(merged2["count"] + 1)

# absolute distance from midpoint
SCALE_MIDPOINT = merged2["response_mean"].mean()
merged2["abs_dist_midpoint"] = (
    merged2["response_mean"] - SCALE_MIDPOINT
).abs()

result = (
    merged2
    .groupby(["countryname", "langname", "id"])
    .agg(
        corr_logcount_response_mean=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_mean"]
            )
        ),
        corr_logcount_response_sd=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_std"]
            )
        ),
        corr_logcount_abs_dist_midpoint=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "abs_dist_midpoint"]
            )
        ),

    )
    .reset_index()
)


In [2116]:
result

,countryname,langname,id,corr_logcount_response_mean,corr_logcount_response_sd,corr_logcount_abs_dist_midpoint
0,Brazil,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.058824
1,Bulgaria,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.265190
2,Chile,Spanish,umn.31951d01452389x,0.199989,-0.153504,0.290988
3,China,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.116186
4,Colombia,Spanish,umn.31951d01452389x,0.029407,-0.270589,-0.068483
5,Croatia,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.363570,-0.037434,-0.247292
6,Curacao,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.211604
7,Denmark,Danish,uc1.b3832576,-0.322866,-0.191735,0.313704
8,Egypt,Arabic,mdp.39015043036436,-0.114331,0.133291,-0.338161
9,Finland,Finnish,mdp.39015059174667,0.072036,-0.299668,-0.086969


In [ ]:
result.to_csv(Path("~/data/Fundemental-Emotions/output_results/bila_long_noun_lemmatized_full.csv").expanduser())

In [218]:
merged.columns

Index(['langname', 'glottocode', 'word', 'count', 'id', 'year', 'countryname',
       'response_mean', 'std'],
      dtype='object')

In [2123]:
import plotly.graph_objects as go

df_plot = result.copy()
df_plot = df_plot.sort_values('langname')
df_plot["sample"] = df_plot["countryname"] + " – " + df_plot["langname"]


In [2124]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_mean"],
    mode="markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_sd"],
    mode="markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color= 'blue')
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red")
))

fig.update_layout(
    title="Correlations between Log Count and Response Measures",
    yaxis_title="Pearson correlation",
    xaxis_title="Sample (Country – Language)",
    xaxis_tickangle=45,
    height=600,
    legend_title="Correlation type",
    template="simple_white"
)

fig.show()


In [2127]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_mean"],
    mode="lines+markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime"),
    line=dict(width=2, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_sd"],
    mode="lines+markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color="blue"),
    line=dict(width=2, color="blue")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="lines+markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red"),
    line=dict(width=2, color="red")
))

fig.add_hline(
    y=0,
    line_width=2,
    line_color="black"
)


fig.update_layout(
    title="Correlations between Log Count and Response Measures",
    yaxis_title="Pearson correlation",
    xaxis_title="Sample (Country – Language)",
    xaxis_tickangle=45,
    height=700,
    legend_title="Correlation type",
    template="simple_white"
)

fig.show()


In [2130]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_response_mean"],
    mode="markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_response_sd"],
    mode="markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color="blue")
))

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red")
))

# horizontal zero line
fig.add_hline(
    y=0,
    line_width=1,
    line_dash="solid",
    line_color="black"
)

fig.update_layout(
    title="Correlations by Language, Searching for Hypocognition",
    xaxis_title="Language",
    yaxis_title="Pearson correlation",
    xaxis_tickangle=45,
    height=700,
    template="simple_white",
    legend_title="Correlation type"
)

fig.show()


In [ ]:
fig.write_html(Path("reference/lang_names.csv"))